In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# LDA

## 1. Importing Libraries and configuring pandas

In [ ]:
import string

import numpy as np
import pandas as pd
from IPython.display import display

pd.options.plotting.backend = "plotly"
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)
pd.options.mode.copy_on_write = True


## 2. Loading data frames

we need to convert some data

In [ ]:
scopus_frame = pd.read_csv('../input/scopus.csv', dtype={'Cited by': int})
ieee_frame = pd.read_csv('../input/ieee.csv', dtype={'DOI': str, 'ISBN': str})
acm_frame = pd.read_csv('../input/acm.csv', dtype={'Cited by': int})

## 3. Convert columns to be interoperable with each other

In [ ]:
from src.utils.scopus import scopus_cols
from src.utils.data import transform_acm_to_scopus, transform_ieee_to_scopus

merged_df = pd.concat([scopus_frame.filter(scopus_cols), transform_ieee_to_scopus(ieee_frame), transform_acm_to_scopus(acm_frame)])
display(merged_df['Source'].astype(str).unique())
display(merged_df.info())

array(['Scopus', 'IEEE', 'ACM Digital Library'], dtype=object)

<class 'pandas.core.frame.DataFrame'>
Index: 485 entries, 0 to 52
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Source                         485 non-null    object
 1   Title                          485 non-null    object
 2   Year                           485 non-null    int64 
 3   Source title                   310 non-null    object
 4   Cited by                       469 non-null    object
 5   Document Type                  485 non-null    object
 6   Abstract                       483 non-null    object
 7   Author full names              469 non-null    object
 8   DOI                            420 non-null    object
 9   Index Keywords                 455 non-null    object
 10  Author Keywords                383 non-null    object
 11  Volume                         225 non-null    object
 12  Language of Original Document  110 non-null    object
 13  Publisher  

None

## 4. Now check for duplicate entries

In [ ]:
column = 'Title'
v = merged_df[column].str.lower().value_counts()
duplicates = merged_df[merged_df[column].str.lower().isin(v.index[v.gt(1)])].sort_values(column)
display(len(duplicates))

159

## 5. "Normalize" author names

In [ ]:
copy = merged_df.copy()
copy['Author full names'] = copy['Author full names'].str.replace(r' \([0-9]*\)', '', regex=True)
print("Length:", copy.shape[0])
copy['Author full names'] = copy['Author full names'].str.split(';')
print("Length:", copy.shape[0])
copy["Title"] = copy['Title'].str.replace('[{}]'.format(string.punctuation), '')
display(copy['Source'].astype(str).unique())
normalized_merged_df = copy.copy()

Length: 485
Length: 485


array(['Scopus', 'IEEE', 'ACM Digital Library'], dtype=object)

## 5. Group duplicates (DOI and Title)

In [ ]:

# Function to flatten lists only if multiple exist
def flatten_if_multiple(series):
    #if series.size > 1:
    #series.describe()
    #print(series.name, series.dropna().values)
    v = series.dropna().values
    
    if v.size == 0:
        return np.nan
    else:
        temp_v = []
        for i in v:
            if type(i) == list:
                temp_v += i
            else:
                temp_v.append(i)
        v = np.array(temp_v)
    if series.name == "Author full names":
        #print(series.name, v)
        test = ""
    if not series.name in ["Abstract", "Source title", "Language of Original Document", "Title", "Document Type"]:
        #if v.size > 1:
            #print(v, v.size)
        
        if v.size == 1:
            return v[0]
        else:
            # print(v)
            # print(type(v))
            # print(type(set(v)))
            l  = list(set(v))
            if series.name in ["Author full names"]: # , "Source"
                #print(series.name, v.size, l)
                return "; ".join(l) if len(l) > 1 else l[0]
            if series.name == "Source":
                #print(series.name, v.size, l)
                
                return min(v)
            if series.name == "Year":
                return max(l) if len(l) > 1 else l[0]
            
            if series.name == "DOI":
                #print(series.name, v.size, l)
                return min(l) if len(l) > 1 else l[0]
            
            if series.name == "Cited by":
                arr = pd.to_numeric(v, errors="coerce")
                l = arr[~np.isnan(arr)]
                if len(l) > 1:
                    return l.sum()
                else:
                    return 0 if len(l) == 0 else l[0]
            
            return l if len(l) > 1 else l[0]
    else:
        #print("Biggest value: ", v)
        return max(set(v), key=len)
        #return series.values[0]
#from src.utils.data import flatten_if_multiple


def group_by_column(column, copy):
    # Define aggregation functions dynamically
    aggregation_functions = {
        col: flatten_if_multiple for col in copy.columns if col != column
    }
    
    #print(aggregation_functions)
    
    copy_with_column = copy[copy[column].notna()]
    grouped_by_copy_with_column = copy_with_column.groupby(column, as_index=False).agg(aggregation_functions)
    copy_without_column = copy[copy[column].isna()]
    

    print(f"Rows with {column}: ", copy_with_column.shape[0])
    print(f"Rows without {column}: ", copy_without_column.shape[0])
    print(f"Rows with {column} grouped: ", grouped_by_copy_with_column.shape[0])
    removed_redundant_column = pd.concat([grouped_by_copy_with_column, copy_without_column], ignore_index=True)
    #removed_redundant_dois["Title"] = removed_redundant_dois['Title'].str.replace('[{}]'.format(string.punctuation), '')
    print(f"Rows without {column} and grouped {column}", removed_redundant_column.shape[0])
    return removed_redundant_column

In [ ]:
display(copy['Source'].astype(str).unique())
#display(len(copy))
removed_redundant_dois = group_by_column('DOI', normalized_merged_df.copy())
display(f"Removed redundant DOIs: {len(merged_df) - len(removed_redundant_dois)}")
display(removed_redundant_dois['Source'].astype(str).unique())
v = removed_redundant_dois["Title"].value_counts()
print("Rows which's title occures more than once: ", v[v >= 2].shape[0])
    
removed_redundant_titles = group_by_column('Title', removed_redundant_dois.copy())
display(f"Removed redundant Titles: {len(removed_redundant_dois) - len(removed_redundant_titles)}")

grouped_merged_df = removed_redundant_titles.copy()
display(f"Length of final dataset: {len(grouped_merged_df)}")

array(['Scopus', 'IEEE', 'ACM Digital Library'], dtype=object)

Rows with DOI:  420
Rows without DOI:  65
Rows with DOI grouped:  344
Rows without DOI and grouped DOI 409


'Removed redundant DOIs: 76'

array(['Scopus', 'ACM Digital Library', 'IEEE'], dtype=object)

Rows which's title occures more than once:  8
Rows with Title:  409
Rows without Title:  0
Rows with Title grouped:  399
Rows without Title and grouped Title 399


'Removed redundant Titles: 10'

'Length of final dataset: 399'

## 6 Final cleanup of data frame

The abstracts which i probably use later on contains phrases like "© 2025 Elsevier B.V., All rights reserved."  which were used for topic creation in a previous iteration. I tried to remove them as stop words and removed all words that occure in more than 90% of the coduments. Additionally i need to transform the abstracts in order to remove publisher specific strings.

In [ ]:
grouped_merged_df['Abstract'] = grouped_merged_df['Abstract'].str.replace('© [0-9]{4} Elsevier B.V., All rights reserved.', repl="", regex=True)

## 7. Final display of preparation and saving to csv

In [ ]:
import csv
from pathlib import Path

# quoting=csv.QUOTE_ALL

grouped_merged_df["Cited by"].astype(str)
display(grouped_merged_df.info())

folder = Path("../output")
folder.mkdir(parents=True, exist_ok=True)

grouped_merged_df.to_csv('../output/merged.csv', quoting = csv.QUOTE_NONNUMERIC)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 399 entries, 0 to 398
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Title                          399 non-null    object
 1   DOI                            343 non-null    object
 2   Source                         399 non-null    object
 3   Year                           399 non-null    int64 
 4   Source title                   271 non-null    object
 5   Cited by                       396 non-null    object
 6   Document Type                  399 non-null    object
 7   Abstract                       399 non-null    object
 8   Author full names              387 non-null    object
 9   Index Keywords                 375 non-null    object
 10  Author Keywords                317 non-null    object
 11  Volume                         201 non-null    object
 12  Language of Original Document  95 non-null     object
 13  Publi

None